In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import EMNIST
from torch.utils.data import DataLoader

# Define transforms - COMPLETE THE MISSING PARTS
transform = transforms.Compose([
    transforms.Resize((28, 28)),
    transforms.Grayscale(3),  # Convert grayscale to RGB (Don't Touch!!)
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                          std=[0.229, 0.224, 0.225])
])

# Load EMNIST letters dataset (given)
train_dataset = EMNIST(root='./data', split='letters', train=True, download=True, transform=transform)
test_dataset = EMNIST(root='./data', split='letters', train=False, download=True, transform=transform)

# Note: EMNIST letters has labels 1-26 (A-Z), so we have 26 classes
num_classes = 26
print(f"Training samples: {len(train_dataset)}")
print(f"Testing samples: {len(test_dataset)}")
print(f"Number of classes: {num_classes}")

In [ ]:
# Letter mapping (labels are 1-26 for A-Z)
letters = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ'
class_labels = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4,"F": 5, "G": 6, "H": 7, "I": 8, "J": 9,"K": 10, "L": 11, "M": 12, "N": 13, "O": 14,"P": 15, "Q": 16, "R": 17, "S": 18, "T": 19, "U": 20,"V": 21, "W": 22, "X": 23, "Y": 24, "Z": 25}
print(class_labels)
# Create DataLoaders and display samples
# Write your code here

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True, num_workers=2)

import matplotlib.pyplot as plt
import numpy as np


mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])


fig, axes = plt.subplots(1, 6, figsize=(15, 5))

imgs_indices = [270,233,110,89,15,20]

for i in range(6):
    img, label = train_dataset[imgs_indices[i]]  #


    img_np = img.numpy().transpose(1, 2, 0)  # (C, H, W) → (H, W, C)


    img_np = std * img_np + mean
    img_np = np.clip(img_np, 0, 1)

    # Show image
    axes[i].imshow(img_np)
    axes[i].axis('off')

plt.show()

In [ ]:
class_labels = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4,"F": 5, "G": 6, "H": 7, "I": 8, "J": 9,"K": 10, "L": 11, "M": 12, "N": 13, "O": 14,"P": 15, "Q": 16, "R": 17, "S": 18, "T": 19, "U": 20,"V": 21, "W": 22, "X": 23, "Y": 24, "Z": 25}
print(class_labels)


image_paths = []
labels = []
import glob
for class_name, label in class_labels.items():
        class_images = glob.glob(f"{'./data'}/train/{class_name}/*")  # Find all images
        image_paths.extend(class_images)
        labels.extend([label] * len(class_images))  # Assign labels

In [ ]:
images, labels = next(iter(train_loader))
print(f"Batch shape: {images.shape}, Labels: {labels}")

In [ ]:
import torch.nn as nn
from torchvision.models import efficientnet_v2_s

# Write your code here
# Load pretrained EfficientNet-B0
model = efficientnet_v2_s(pretrained=True)

# Modify the classifier for 26 classes
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 26)

# Move to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print(model)

In [ ]:
# Write your code here
from tqdm import tqdm    # Shows progress bar
print(labels)
# Training Loop
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in tqdm(dataloader):
        images, labels = images.to(device), labels.to(device)


        outputs = model(images).squeeze()  # The model outputs in shape [batch_size,1]. We convert them to [batch_size,] so the loss accepts them.
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # Track accuracy
        predictions = torch.sigmoid(outputs) > 0.5  # Get predicted class
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy

# Validation Loop
def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images).squeeze()  # The model outputs in shape [batch_size,1]. We convert them to [batch_size,] so the loss accepts them.
            loss = criterion(outputs, labels)
            total_loss += loss.item()

            # Compute accuracy
            predictions = torch.sigmoid(outputs) > 0.5  # Get predicted class
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy


In [ ]:
import torch.optim as optim

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)
num_epochs = 20 # Number of epochs


# Lists to store metrics
train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

# Training process
for epoch in range(num_epochs):
    train_loss, train_accuracy = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_accuracy = validate(model, test_loader, criterion, device)

    # Store metrics
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_accuracy)
    val_accuracies.append(val_accuracy)

    print(f"Epoch {epoch+1}/{num_epochs}: "
          f"Train Loss={train_loss:.4f}, Train Accuracy={train_accuracy:.2f}%, "
          f"Val Loss={val_loss:.4f}, Val Accuracy={val_accuracy:.2f}%")


In [ ]:
# Write your code here
